# 11 — Build Hourly Workload Model Input

## Research purpose

Combine campus capacity, renewable availability rankings and the generic UKPN load shape into the hourly input for workload shifting. Apply the load shape on each location's local clock while retaining UTC for renewable rankings and daily allocation boundaries.


## Imports

Load the libraries used to join campus capacity, renewable rankings and load profiles.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## Project location and paths

Locate the project folder and define the paths for the campus, renewable and load-profile inputs. Define where to save the hourly model input, daily diagnostics and workload scenarios.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "notebooks").is_dir() or not (PROJECT_ROOT / "data").is_dir():
    raise RuntimeError("Launch Jupyter from the folder containing data/ and notebooks/, or from its notebooks/ directory.")

DATA_MANUAL = PROJECT_ROOT / "data" / "manual"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_FINAL = PROJECT_ROOT / "data" / "final"
CAMPUSES_PATH = DATA_FINAL / "hyperscale_campuses_current_plus_future_scenario.csv"
QUERY_LOCATIONS_PATH = DATA_INTERIM / "renewables_ninja_query_locations.csv"
CAMPUS_MAPPING_PATH = DATA_INTERIM / "renewables_ninja_campus_query_mapping.csv"
RENEWABLE_RANKINGS_PATH = DATA_INTERIM / "renewable_availability_cluster_ranked_hourly.parquet"
LOAD_PROFILES_PATH = DATA_INTERIM / "datacenter_generic_load_shape_profiles.csv"
HOURLY_OUTPUT_PATH = DATA_INTERIM / "cluster_operator_hourly_load_and_flexibility.parquet"
DAILY_OUTPUT_PATH = DATA_INTERIM / "cluster_operator_daily_flexibility_summary.csv"
SCENARIOS_OUTPUT_PATH = DATA_INTERIM / "flexibility_scenarios.csv"
UTILISATION_SCENARIOS_OUTPUT_PATH = DATA_INTERIM / "utilisation_scenarios.csv"

## Settings and assumptions

Use weather years 2023–2025, the all-site UKPN profile and a receiver utilisation ceiling of 95%. Apply the profile by each node's local clock, account for Kolkata's half-hour offset and restrict the network to transfers within each operator.


In [3]:
ACTIVE_WEATHER_YEARS = [2023, 2024, 2025]
RECEIVE_UTILISATION_CEILING = 0.95
LOAD_PROFILE_VARIANT = "all_clean_sites"
LOAD_PROFILE_TIME_BASIS = "node_local_clock"
HALF_HOUR_OVERLAP_TIMEZONE = "Asia/Kolkata"
NETWORK_SCOPE = "operator_only"

## Read and validate workload scenarios

Check that the manual scenario table contains all combinations of 80%, 85% and 90% utilisation with 20%, 25% and 30% flexibility. Select 85% utilisation and 25% flexibility for the hourly diagnostics and prepare the separate scenario tables used by the allocation model.


In [4]:
SCENARIO_MATRIX = pd.read_csv(DATA_MANUAL / "workload_scenarios.csv")

required_scenario_columns = {
    "utilisation_scenario_id", "base_utilisation", "utilisation_scenario_label",
    "flexibility_scenario_id", "flexible_workload_share", "flexibility_scenario_label",
    "is_central_utilisation", "is_central_flexibility", "is_central",
}

missing = sorted(required_scenario_columns - set(SCENARIO_MATRIX.columns))
if missing:
    raise ValueError(f"Manual workload scenarios are missing columns: {missing}")
for column in ["is_central_utilisation", "is_central_flexibility", "is_central"]:
    values = SCENARIO_MATRIX[column].astype(str).str.lower()
    if not values.isin(["true", "false"]).all():
        raise ValueError(f"Workload scenario {column} must contain Boolean values")
    SCENARIO_MATRIX[column] = values.eq("true")

scenario_keys = ["base_utilisation", "flexible_workload_share"]
expected_matrix = pd.MultiIndex.from_product([[0.80, 0.85, 0.90], [0.20, 0.25, 0.30]])

actual_matrix = pd.MultiIndex.from_frame(SCENARIO_MATRIX[scenario_keys])
if actual_matrix.has_duplicates or set(actual_matrix) != set(expected_matrix):
    raise ValueError("Workload scenarios must contain the full 3 × 3 utilisation/flexibility matrix")
for flag, expected in [
    ("is_central_utilisation", SCENARIO_MATRIX["base_utilisation"].eq(0.85)),
    ("is_central_flexibility", SCENARIO_MATRIX["flexible_workload_share"].eq(0.25)),
    ("is_central", SCENARIO_MATRIX["base_utilisation"].eq(0.85) & SCENARIO_MATRIX["flexible_workload_share"].eq(0.25)),
]:
    if not SCENARIO_MATRIX[flag].equals(expected):
        raise ValueError(f"Workload scenario {flag} must identify the 85% / 25% reference")
UTILISATION_SCENARIOS = (
    SCENARIO_MATRIX[[
        "utilisation_scenario_id", "base_utilisation",
        "utilisation_scenario_label", "is_central_utilisation",
    ]]
    .drop_duplicates()
    .rename(columns={
        "utilisation_scenario_label": "scenario_label",
        "is_central_utilisation": "is_central",
    })
)
FLEXIBILITY_SCENARIOS = (
    SCENARIO_MATRIX[[
        "flexibility_scenario_id", "flexible_workload_share",
        "flexibility_scenario_label", "is_central_flexibility",
    ]]
    .drop_duplicates()
    .rename(columns={
        "flexibility_scenario_label": "scenario_label",
        "is_central_flexibility": "is_central",
    })
)
reference_scenario = SCENARIO_MATRIX.loc[SCENARIO_MATRIX["is_central"]].iloc[0]
BASE_UTILISATION = float(reference_scenario["base_utilisation"])
FLEXIBLE_WORKLOAD_SHARE = float(reference_scenario["flexible_workload_share"])

SCENARIO_MATRIX

,utilisation_scenario_id,base_utilisation,utilisation_scenario_label,is_central_utilisation,flexibility_scenario_id,flexible_workload_share,flexibility_scenario_label,is_central_flexibility,is_central
0,util_80,0.80,Low 80% baseline utilisation,False,flex_20,0.20,Conservative 20% flexible workload,False,False
1,util_80,0.80,Low 80% baseline utilisation,False,flex_25,0.25,Central 25% flexible workload,True,False
2,util_80,0.80,Low 80% baseline utilisation,False,flex_30,0.30,High 30% flexible workload,False,False
3,util_85,0.85,Central 85% baseline utilisation,True,flex_20,0.20,Conservative 20% flexible workload,False,False
4,util_85,0.85,Central 85% baseline utilisation,True,flex_25,0.25,Central 25% flexible workload,True,True
5,util_85,0.85,Central 85% baseline utilisation,True,flex_30,0.30,High 30% flexible workload,False,False
6,util_90,0.90,High 90% baseline utilisation,False,flex_20,0.20,Conservative 20% flexible workload,False,False
7,util_90,0.90,High 90% baseline utilisation,False,flex_25,0.25,Central 25% flexible workload,True,False
8,util_90,0.90,High 90% baseline utilisation,False,flex_30,0.30,High 30% flexible workload,False,False


## Load and validate the inputs

Read the campus, query-location, renewable-ranking and load-profile tables and check their required fields. Retain the selected weather years and attach each query location's IANA timezone, rejecting missing or inconsistent mappings. Record the UTC ranking fields to check that the local-clock join preserves them.


In [5]:
campuses = pd.read_csv(CAMPUSES_PATH)
query_locations = pd.read_csv(QUERY_LOCATIONS_PATH)
renewable = pd.read_parquet(RENEWABLE_RANKINGS_PATH)
profiles = pd.read_csv(LOAD_PROFILES_PATH)

required_campus_cols = {
    "campus_group_id", "operator_group", "country", "country_code", "world_region", "sub_region",
    "timezone", "latitude", "longitude", "record_layer", "status", "capacity_mw", "it_load_mw",
}
required_query_cols = {
    "query_location_id", "country_code", "country", "world_region", "sub_region", "timezone",
    "renewable_query_region", "representative_lat", "representative_lon",
    "cluster_radius_km", "query_method",
}
required_renewable_cols = {
    "query_location_id", "weather_year", "time_utc", "utc_date", "utc_hour",
    "day_of_week_utc", "is_weekend_utc", "solar_capacity_factor", "wind_capacity_factor",
    "renewable_availability_index", "receive_rank_daily", "send_rank_daily",
    "receive_candidate", "send_candidate",
}
required_profile_cols = {
    "profile_variant", "profile_timezone", "profile_month", "profile_day_type", "profile_hour", "normalized_load_shape",
}
for label, df, required in [
    ("campuses", campuses, required_campus_cols),
    ("query_locations", query_locations, required_query_cols),
    ("renewable", renewable, required_renewable_cols),
    ("profiles", profiles, required_profile_cols),
]:
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"{label} is missing required columns: {missing}")

for column in ["capacity_mw", "it_load_mw"]:
    campuses[column] = pd.to_numeric(campuses[column], errors="coerce")

renewable["weather_year"] = pd.to_numeric(renewable["weather_year"], errors="coerce").astype("Int64")
renewable = renewable[renewable["weather_year"].isin(ACTIVE_WEATHER_YEARS)].copy()
renewable["time_utc"] = pd.to_datetime(renewable["time_utc"], utc=True, errors="coerce")
renewable["utc_hour"] = pd.to_numeric(renewable["utc_hour"], errors="coerce").astype("Int64")
renewable["utc_month"] = renewable["time_utc"].dt.month.astype("Int64")
renewable["utc_day_type"] = np.where(renewable["is_weekend_utc"].astype(bool), "weekend", "weekday")

timezone_lookup = query_locations.set_index("query_location_id")["timezone"]
mapped_timezones = renewable["query_location_id"].map(timezone_lookup)
if mapped_timezones.isna().any():
    raise ValueError("At least one renewable row has no query-location IANA timezone")
if "timezone" in renewable.columns and not renewable["timezone"].astype(str).equals(mapped_timezones.astype(str)):
    raise ValueError("Renewable and query-location timezone fields disagree")
renewable["timezone"] = mapped_timezones
ranking_invariant_columns = [
    "query_location_id", "weather_year", "time_utc", "utc_date", "utc_hour",
    "receive_rank_daily", "send_rank_daily", "receive_candidate", "send_candidate",
]

ranking_signature_before = int(pd.util.hash_pandas_object(renewable[ranking_invariant_columns], index=False).sum())

print(f"Campuses: {len(campuses):,}")
print(f"Query locations: {len(query_locations):,}")
print(f"Renewable hourly rows: {len(renewable):,}")
print(f"Weather years: {sorted(renewable['weather_year'].dropna().astype(int).unique().tolist())}")
print(f"Load profile variants: {sorted(profiles['profile_variant'].dropna().unique().tolist())}")

Campuses: 202
Query locations: 87
Renewable hourly rows: 2,288,448
Weather years: [2023, 2024, 2025]
Load profile variants: ['all_clean_sites', 'co_located', 'enterprise']


## Build current and current-plus-future layers

Join campuses to their renewable query locations and retain campuses with positive known capacity. Build the current-only and current-plus-future layers, then sum capacity by operator and query location. Keep campus identifiers, source layers and status labels with the aggregates.


In [6]:
query_campus_map = pd.read_csv(CAMPUS_MAPPING_PATH)[
    ["campus_group_id", "query_location_id", "distance_to_representative_km"]
].merge(
    query_locations.drop(columns=["campus_refs"], errors="ignore"),
    on="query_location_id", how="left", validate="many_to_one"
)
campus_join = query_campus_map.merge(
    campuses,
    on="campus_group_id",
    how="left",
    suffixes=("_query", "_campus"),
    validate="many_to_one",
)

missing_campus = campus_join["operator_group"].isna().sum()
if missing_campus:
    missing_refs = sorted(campus_join.loc[campus_join["operator_group"].isna(), "campus_group_id"].dropna().unique().tolist())[:10]
    raise ValueError(f"{missing_campus:,} query-campus references did not match campus records. Examples: {missing_refs}")

campus_join = campus_join[campus_join["capacity_mw"].notna() & campus_join["capacity_mw"].gt(0)].copy()
campus_join["it_load_mw"] = campus_join["it_load_mw"].fillna(0)

current_only = campus_join[campus_join["record_layer"].eq("current_operational_osm")].copy()
current_only["model_layer"] = "current_only"
current_plus_future = campus_join.copy()
current_plus_future["model_layer"] = "current_plus_future"
model_campuses = pd.concat([current_only, current_plus_future], ignore_index=True)

if model_campuses.empty:
    raise ValueError("No campus records available for modelling")

def join_unique(values):
    clean = sorted({str(value) for value in values if pd.notna(value) and str(value) != ""})
    return "|".join(clean)

cluster_operator_capacity = (
    model_campuses.groupby(
        [
            "model_layer", "query_location_id", "operator_group", "country_code_query", "country_query",
            "world_region_query", "sub_region_query", "timezone_query", "renewable_query_region",
            "representative_lat", "representative_lon", "cluster_radius_km", "query_method",
        ],
        dropna=False,
    )
    .agg(
        campus_count=("campus_group_id", "nunique"),
        current_campus_count=("record_layer", lambda s: int((s == "current_operational_osm").sum())),
        future_campus_count=("record_layer", lambda s: int((s == "future_operator_source").sum())),
        capacity_mw=("capacity_mw", "sum"),
        it_load_mw=("it_load_mw", "sum"),
        campus_group_ids=("campus_group_id", join_unique),
        record_layers=("record_layer", join_unique),
        statuses=("status", join_unique),
    )
    .reset_index()
    .rename(
        columns={
            "country_code_query": "country_code",
            "country_query": "country",
            "world_region_query": "world_region",
            "sub_region_query": "sub_region",
            "timezone_query": "timezone",
        }
    )
)
cluster_operator_capacity = cluster_operator_capacity[cluster_operator_capacity["capacity_mw"].gt(0)].copy()

print("Model layers:")
print(cluster_operator_capacity["model_layer"].value_counts().to_string())
print("Operator rows by layer:")
print(cluster_operator_capacity.groupby(["model_layer", "operator_group"]).size().to_string())
cluster_operator_capacity.head()

Model layers:
model_layer
current_plus_future    109
current_only            78
Operator rows by layer:
model_layer          operator_group
current_only         Amazon            26
                     Google            22
                     Microsoft         30
current_plus_future  Amazon            28
                     Google            42
                     Microsoft         39


,model_layer,query_location_id,operator_group,country_code,country,world_region,sub_region,timezone,renewable_query_region,representative_lat,...,cluster_radius_km,query_method,campus_count,current_campus_count,future_campus_count,capacity_mw,it_load_mw,campus_group_ids,record_layers,statuses
0,current_only,RN-ARE-DUBAI-01,Amazon,ARE,United Arab Emirates,Middle East and Africa,Western Asia,Asia/Dubai,ARE Asia/Dubai cluster 01,24.631681,...,100.0,capacity_weighted_centroid_within_100km_cluster,3,3,0,60.685811,52.770270,CAMPUS-00107|CAMPUS-00161|CAMPUS-00162,current_operational_osm,operational
1,current_only,RN-AUS-MELBOURNE-01,Amazon,AUS,Australia,Asia Pacific,Australia and New Zealand,Australia/Melbourne,AUS Australia/Melbourne cluster 01,-37.780600,...,100.0,capacity_weighted_centroid_within_100km_cluster,1,1,0,9.643418,8.385581,CAMPUS-00155,current_operational_osm,operational
2,current_only,RN-AUS-MELBOURNE-01,Microsoft,AUS,Australia,Asia Pacific,Australia and New Zealand,Australia/Melbourne,AUS Australia/Melbourne cluster 01,-37.780600,...,100.0,capacity_weighted_centroid_within_100km_cluster,3,3,0,101.495408,86.748212,CAMPUS-00136|CAMPUS-00142|CAMPUS-00145,current_operational_osm,operational
3,current_only,RN-AUS-SYDNEY-01,Amazon,AUS,Australia,Asia Pacific,Australia and New Zealand,Australia/Sydney,AUS Australia/Sydney cluster 01,-33.879302,...,100.0,capacity_weighted_centroid_within_100km_cluster,4,4,0,155.144447,134.908215,CAMPUS-00068|CAMPUS-00094|CAMPUS-00140|CAMPUS-...,current_operational_osm,operational
4,current_only,RN-AUS-SYDNEY-01,Microsoft,AUS,Australia,Asia Pacific,Australia and New Zealand,Australia/Sydney,AUS Australia/Sydney cluster 01,-33.879302,...,100.0,capacity_weighted_centroid_within_100km_cluster,1,1,0,70.881364,60.582363,CAMPUS-00139,current_operational_osm,operational


## Convert UTC timestamps to local time

Define a function that adds each node's local date, month, day type, hour, minute and UTC offset using its IANA timezone. Check that timestamps and timezone values are valid before converting them.


In [7]:
def add_node_local_clock_fields(frame: pd.DataFrame) -> pd.DataFrame:
    """Derive auditable local civil-clock fields from canonical UTC timestamps."""
    required = {"time_utc", "timezone"}
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f"Local-clock conversion is missing columns: {missing}")
    result = frame.reset_index(drop=True).copy()
    result["time_utc"] = pd.to_datetime(result["time_utc"], utc=True, errors="coerce")
    if result["time_utc"].isna().any() or result["timezone"].isna().any():
        raise ValueError("Local-clock conversion requires valid UTC timestamps and IANA timezones")

    row_count = len(result)
    local_date = np.empty(row_count, dtype=object)
    local_month = np.empty(row_count, dtype=np.int8)
    local_day_of_week = np.empty(row_count, dtype=np.int8)
    local_hour = np.empty(row_count, dtype=np.int8)
    local_minute = np.empty(row_count, dtype=np.int8)
    utc_offset_minutes = np.empty(row_count, dtype=np.int16)

    for timezone_name, positions in result.groupby("timezone", sort=True).indices.items():
        utc = result.iloc[positions]["time_utc"]
        try:
            local = utc.dt.tz_convert(str(timezone_name))
        except Exception as exc:
            raise ValueError(f"Invalid IANA timezone {timezone_name!r}") from exc
        local_date[positions] = local.dt.date.astype(str).to_numpy()
        local_month[positions] = local.dt.month.to_numpy(dtype=np.int8)
        local_day_of_week[positions] = local.dt.dayofweek.to_numpy(dtype=np.int8)
        local_hour[positions] = local.dt.hour.to_numpy(dtype=np.int8)
        local_minute[positions] = local.dt.minute.to_numpy(dtype=np.int8)
        local_naive = local.dt.tz_localize(None).to_numpy(dtype="datetime64[ns]")
        utc_naive = utc.dt.tz_localize(None).to_numpy(dtype="datetime64[ns]")
        utc_offset_minutes[positions] = ((local_naive - utc_naive) / np.timedelta64(1, "m")).astype(np.int16)

    result["local_date"] = local_date
    result["local_month"] = local_month
    result["local_day_of_week"] = local_day_of_week
    result["local_day_type"] = np.where(np.isin(local_day_of_week, [5, 6]), "weekend", "weekday")
    result["local_hour"] = local_hour
    result["local_minute"] = local_minute
    result["utc_offset_minutes"] = utc_offset_minutes
    return result

## Look up the local load shape

Define the load-shape lookup using local month, weekday or weekend, and hour. Require every lookup key to match a value in the UKPN profile grid.


In [8]:
def lookup_profile_shape(
    profile_lookup: pd.Series,
    frame: pd.DataFrame,
) -> np.ndarray:
    keys = pd.MultiIndex.from_arrays(
        [frame["local_month"], frame["local_day_type"], frame["local_hour"]],
        names=["profile_month", "profile_day_type", "profile_hour"],
    )
    values = profile_lookup.reindex(keys).to_numpy(dtype=float)
    if np.isnan(values).any():
        raise ValueError("At least one local clock did not match the complete UKPN profile grid")
    return values

## Account for hourly interval overlap

Define how to attach the local load shape to each hourly renewable record. Use one profile bin for whole-hour offsets and equal weights from the two intersected bins for Kolkata's half-hour offset, including intervals that cross date or month boundaries.


In [9]:
def attach_node_local_load_shape(
    ranked_hourly: pd.DataFrame,
    profile: pd.DataFrame,
) -> pd.DataFrame:
    """Attach local-clock shape; overlap-weight Asia/Kolkata's half-hour intervals."""
    profile_keys = ["profile_month", "profile_day_type", "profile_hour"]
    if profile.duplicated(profile_keys).any():
        raise ValueError("Selected UKPN profile has duplicate clock keys")
    profile_lookup = profile.set_index(profile_keys)["normalized_load_shape"].sort_index()
    result = add_node_local_clock_fields(ranked_hourly)
    primary = lookup_profile_shape(
        profile_lookup, result
    )
    result["normalized_load_shape"] = primary
    result["load_profile_join_method"] = "single_local_hour"

    overlap_mask = result["timezone"].eq(HALF_HOUR_OVERLAP_TIMEZONE)
    if not result.loc[~overlap_mask, "local_minute"].eq(0).all():
        unexpected = sorted(result.loc[~overlap_mask & result["local_minute"].ne(0), "timezone"].unique().tolist())
        raise ValueError(f"Non-integral local-hour offsets require an explicit overlap rule: {unexpected}")
    if overlap_mask.any():
        if not result.loc[overlap_mask, "local_minute"].eq(30).all():
            raise ValueError("Asia/Kolkata rows must start at local minute 30")
        secondary_source = result.loc[overlap_mask, ["time_utc", "timezone"]].copy()
        secondary_source["time_utc"] = secondary_source["time_utc"] + pd.Timedelta(minutes=30)
        secondary_clock = add_node_local_clock_fields(secondary_source)
        secondary = lookup_profile_shape(
            profile_lookup, secondary_clock
        )
        result.loc[overlap_mask, "normalized_load_shape"] = 0.5 * (primary[overlap_mask.to_numpy()] + secondary)
        result.loc[overlap_mask, "load_profile_join_method"] = "half_hour_overlap_50_50"

    return result

## Join rankings, load shape and capacity

Select the all-site UKPN profile and attach its normalised shape using each node's local clock. Check that UTC timestamps, daily ranks and candidate flags are unchanged, then join the hourly series to each model layer's operator and location capacity.


In [10]:
profile = profiles[profiles["profile_variant"].eq(LOAD_PROFILE_VARIANT)].copy()
if profile.empty:
    raise ValueError(f"Profile variant not found: {LOAD_PROFILE_VARIANT}")
if set(profile["profile_timezone"].dropna().unique()) != {"Europe/London"}:
    raise ValueError("The generic UKPN profile must be built on the Europe/London local clock")
profile = profile[["profile_month", "profile_day_type", "profile_hour", "normalized_load_shape"]].copy()
profile["profile_month"] = pd.to_numeric(profile["profile_month"], errors="coerce").astype("Int64")
profile["profile_hour"] = pd.to_numeric(profile["profile_hour"], errors="coerce").astype("Int64")

renewable_keep_cols = [
    "query_location_id", "timezone", "weather_year", "time_utc", "utc_date", "utc_hour", "utc_month",
    "day_of_week_utc", "utc_day_type", "solar_capacity_factor", "wind_capacity_factor",
    "renewable_availability_index", "receive_rank_daily", "send_rank_daily",
    "receive_candidate", "send_candidate",
]

renewable_model = attach_node_local_load_shape(renewable[renewable_keep_cols], profile)
ranking_signature_after = int(pd.util.hash_pandas_object(renewable_model[ranking_invariant_columns], index=False).sum())
if ranking_signature_after != ranking_signature_before:
    raise ValueError("Local-clock attachment changed UTC renewable ranks, timestamps, or candidate flags")

# Keep the detailed campus inventory above; carry only model fields into every hour.
capacity_model = cluster_operator_capacity.drop(columns=[
    "current_campus_count", "future_campus_count", "campus_group_ids", "record_layers",
    "statuses", "cluster_radius_km", "query_method",
])

hourly = capacity_model.merge(
    renewable_model,
    on=["query_location_id", "timezone"],
    how="inner",
    validate="many_to_many",
)

if hourly.empty:
    raise ValueError("No rows after joining cluster/operator capacity to renewable rankings")

for column in [
    "solar_capacity_factor", "wind_capacity_factor", "renewable_availability_index",
    "normalized_load_shape", "capacity_mw", "it_load_mw",
]:
    hourly[column] = pd.to_numeric(hourly[column], errors="coerce")

print(f"Hourly model rows: {len(hourly):,}")
print("Rows by model layer:")
print(hourly["model_layer"].value_counts().to_string())
hourly.head()

Hourly model rows: 4,918,848
Rows by model layer:
model_layer
current_plus_future    2867136
current_only           2051712


,model_layer,query_location_id,operator_group,country_code,country,world_region,sub_region,timezone,renewable_query_region,representative_lat,...,send_candidate,local_date,local_month,local_day_of_week,local_day_type,local_hour,local_minute,utc_offset_minutes,normalized_load_shape,load_profile_join_method
0,current_only,RN-ARE-DUBAI-01,Amazon,ARE,United Arab Emirates,Middle East and Africa,Western Asia,Asia/Dubai,ARE Asia/Dubai cluster 01,24.631681,...,False,2023-01-01,1,6,weekend,4,0,240,0.997370,single_local_hour
1,current_only,RN-ARE-DUBAI-01,Amazon,ARE,United Arab Emirates,Middle East and Africa,Western Asia,Asia/Dubai,ARE Asia/Dubai cluster 01,24.631681,...,False,2023-01-01,1,6,weekend,5,0,240,0.996363,single_local_hour
2,current_only,RN-ARE-DUBAI-01,Amazon,ARE,United Arab Emirates,Middle East and Africa,Western Asia,Asia/Dubai,ARE Asia/Dubai cluster 01,24.631681,...,False,2023-01-01,1,6,weekend,6,0,240,0.995727,single_local_hour
3,current_only,RN-ARE-DUBAI-01,Amazon,ARE,United Arab Emirates,Middle East and Africa,Western Asia,Asia/Dubai,ARE Asia/Dubai cluster 01,24.631681,...,False,2023-01-01,1,6,weekend,7,0,240,0.996671,single_local_hour
4,current_only,RN-ARE-DUBAI-01,Amazon,ARE,United Arab Emirates,Middle East and Africa,Western Asia,Asia/Dubai,ARE Asia/Dubai cluster 01,24.631681,...,False,2023-01-01,1,6,weekend,8,0,240,0.994000,single_local_hour


## Calculate reference load and headroom

Calculate reference load as the smaller of installed capacity and capacity × 0.85 × normalised shape. Set sender potential to 25% of actual load and receiver headroom to the positive difference between 95% of capacity and actual load, using the supplied six-hour flags to select eligible hours. Retain these reference diagnostics alongside the inputs used to recalculate all workload cases in notebook 12.


In [11]:
hourly["utilisation_scenario_id"] = reference_scenario["utilisation_scenario_id"]
hourly["base_utilisation"] = BASE_UTILISATION
hourly["flexible_workload_share"] = FLEXIBLE_WORKLOAD_SHARE
hourly["receive_utilisation_ceiling"] = RECEIVE_UTILISATION_CEILING
hourly["load_profile_variant"] = LOAD_PROFILE_VARIANT
hourly["load_profile_time_basis"] = LOAD_PROFILE_TIME_BASIS
hourly["network_scope"] = NETWORK_SCOPE

hourly["actual_total_load_mw_uncapped"] = hourly["capacity_mw"] * BASE_UTILISATION * hourly["normalized_load_shape"]
hourly["actual_it_load_mw_uncapped"] = hourly["it_load_mw"] * BASE_UTILISATION * hourly["normalized_load_shape"]
hourly["actual_total_load_mw"] = np.minimum(hourly["actual_total_load_mw_uncapped"], hourly["capacity_mw"])
hourly["actual_it_load_mw"] = np.minimum(hourly["actual_it_load_mw_uncapped"], hourly["it_load_mw"])
hourly["load_capped"] = hourly["actual_total_load_mw_uncapped"].gt(hourly["capacity_mw"])

hourly["send_potential_mw"] = hourly["actual_total_load_mw"] * FLEXIBLE_WORKLOAD_SHARE
hourly["receive_headroom_mw"] = (hourly["capacity_mw"] * RECEIVE_UTILISATION_CEILING - hourly["actual_total_load_mw"]).clip(lower=0)
hourly["eligible_send_mw"] = np.where(hourly["send_candidate"].astype(bool), hourly["send_potential_mw"], 0.0)
hourly["eligible_receive_mw"] = np.where(hourly["receive_candidate"].astype(bool), hourly["receive_headroom_mw"], 0.0)

hourly["eligible_send_mwh"] = hourly["eligible_send_mw"]
hourly["eligible_receive_mwh"] = hourly["eligible_receive_mw"]

ordered_cols = [
    "model_layer", "query_location_id", "operator_group", "country", "country_code", "world_region",
    "sub_region", "timezone", "renewable_query_region", "representative_lat", "representative_lon",
    "campus_count", "capacity_mw", "it_load_mw", "weather_year", "time_utc", "utc_date",
    "utc_hour", "utc_month", "day_of_week_utc", "utc_day_type", "local_date", "local_month",
    "local_day_of_week", "local_day_type", "local_hour", "local_minute", "utc_offset_minutes",
    "renewable_availability_index", "receive_rank_daily",
    "send_rank_daily", "receive_candidate", "send_candidate", "normalized_load_shape",
    "load_profile_variant", "load_profile_time_basis", "load_profile_join_method", "utilisation_scenario_id",
    "base_utilisation", "flexible_workload_share", "receive_utilisation_ceiling",
    "network_scope", "actual_total_load_mw", "actual_it_load_mw", "load_capped", "send_potential_mw",
    "receive_headroom_mw", "eligible_send_mw", "eligible_receive_mw", "eligible_send_mwh",
    "eligible_receive_mwh",
]
hourly = hourly[ordered_cols].sort_values(
    ["model_layer", "operator_group", "query_location_id", "weather_year", "time_utc"]
).reset_index(drop=True)

print("Load capped rows:", int(hourly["load_capped"].sum()))
print("Hourly flexibility summary:")
print(hourly[["actual_total_load_mw", "send_potential_mw", "receive_headroom_mw", "eligible_send_mw", "eligible_receive_mw"]].describe().to_string())

Load capped rows: 0
Hourly flexibility summary:
       actual_total_load_mw  send_potential_mw  receive_headroom_mw  eligible_send_mw  eligible_receive_mw
count          4.918848e+06       4.918848e+06         4.918848e+06      4.918848e+06         4.918848e+06
mean           1.996337e+02       4.990843e+01         2.205583e+01      1.233717e+01         4.590499e+00
std            4.008693e+02       1.002173e+02         4.646033e+01      5.394926e+01         2.140665e+01
min            2.640838e-01       6.602095e-02         9.053254e-03      0.000000e+00         0.000000e+00
25%            3.995817e+01       9.989543e+00         3.858999e+00      0.000000e+00         0.000000e+00
50%            8.239494e+01       2.059874e+01         8.385532e+00      0.000000e+00         0.000000e+00
75%            1.619479e+02       4.048697e+01         1.837387e+01      1.650524e-02         2.263314e-03
max            3.284747e+03       8.211868e+02         5.239329e+02      8.211868e+02         5.

## Summarise daily flexibility diagnostics

Aggregate demand, sender potential, receiver headroom and renewable availability by location, operator, model layer and UTC date. Take the smaller of candidate send and receive energy as a local daily bound before network allocation.


In [12]:
daily = (
    hourly.groupby(
        [
            "model_layer", "query_location_id", "operator_group", "country", "country_code", "world_region",
            "sub_region", "timezone", "renewable_query_region", "weather_year", "utc_date",
        ],
        dropna=False,
    )
    .agg(
        capacity_mw=("capacity_mw", "first"),
        it_load_mw=("it_load_mw", "first"),
        campus_count=("campus_count", "first"),
        actual_total_load_mwh=("actual_total_load_mw", "sum"),
        actual_it_load_mwh=("actual_it_load_mw", "sum"),
        daily_send_potential_mwh=("send_potential_mw", "sum"),
        daily_receive_headroom_mwh=("receive_headroom_mw", "sum"),
        daily_send_candidate_mwh=("eligible_send_mwh", "sum"),
        daily_receive_candidate_mwh=("eligible_receive_mwh", "sum"),
        send_candidate_hours=("send_candidate", "sum"),
        receive_candidate_hours=("receive_candidate", "sum"),
        mean_renewable_availability_index=("renewable_availability_index", "mean"),
        min_renewable_availability_index=("renewable_availability_index", "min"),
        max_renewable_availability_index=("renewable_availability_index", "max"),
    )
    .reset_index()
)
daily["daily_balanced_shiftable_mwh"] = daily[["daily_send_candidate_mwh", "daily_receive_candidate_mwh"]].min(axis=1)
daily = daily.sort_values(["model_layer", "operator_group", "query_location_id", "weather_year", "utc_date"]).reset_index(drop=True)

print(f"Daily summary rows: {len(daily):,}")
print("Daily balanced shiftable MWh summary:")
print(daily["daily_balanced_shiftable_mwh"].describe().to_string())

Daily summary rows: 204,952
Daily balanced shiftable MWh summary:
count    204952.000000
mean        110.171974
std         237.064110
min           0.063478
25%          18.680506
50%          40.829349
75%          93.396752
max        3046.387905


## Check local-clock examples

Test selected UTC timestamps against expected local dates, hours, day types and offsets across the model timezones. The examples cover daylight-saving transitions, midnight, leap-day boundaries and Kolkata's half-hour offset.


In [13]:
def validate_node_clock_examples() -> None:
    """Validate standard time, DST, midnight, leap-day, and weekend conversions."""
    cases = pd.DataFrame(
        [
            ("london_winter", "2024-01-15 01:00:00Z", "Europe/London"),
            ("london_summer", "2024-07-15 01:00:00Z", "Europe/London"),
            ("london_spring_before", "2024-03-31 00:00:00Z", "Europe/London"),
            ("london_spring_after", "2024-03-31 01:00:00Z", "Europe/London"),
            ("london_autumn_first", "2024-10-27 00:00:00Z", "Europe/London"),
            ("london_autumn_second", "2024-10-27 01:00:00Z", "Europe/London"),
            ("vienna_winter", "2024-01-15 01:00:00Z", "Europe/Vienna"),
            ("vienna_summer", "2024-07-15 01:00:00Z", "Europe/Vienna"),
            ("vienna_spring_after", "2024-03-31 01:00:00Z", "Europe/Vienna"),
            ("new_york_weekend_boundary", "2024-06-01 00:00:00Z", "America/New_York"),
            ("new_york_leap_boundary", "2024-03-01 00:00:00Z", "America/New_York"),
            ("new_york_spring_before", "2024-03-10 06:00:00Z", "America/New_York"),
            ("new_york_spring_after", "2024-03-10 07:00:00Z", "America/New_York"),
            ("kolkata_half_hour", "2024-01-15 00:00:00Z", "Asia/Kolkata"),
            ("auckland_date_boundary", "2024-01-01 12:00:00Z", "Pacific/Auckland"),
        ],
        columns=["case", "time_utc", "timezone"],
    )
    cases["time_utc"] = pd.to_datetime(cases["time_utc"], utc=True)
    actual = add_node_local_clock_fields(cases)
    expected = [
        ("2024-01-15", 1, 0, "weekday", 0),
        ("2024-07-15", 2, 0, "weekday", 60),
        ("2024-03-31", 0, 0, "weekend", 0),
        ("2024-03-31", 2, 0, "weekend", 60),
        ("2024-10-27", 1, 0, "weekend", 60),
        ("2024-10-27", 1, 0, "weekend", 0),
        ("2024-01-15", 2, 0, "weekday", 60),
        ("2024-07-15", 3, 0, "weekday", 120),
        ("2024-03-31", 3, 0, "weekend", 120),
        ("2024-05-31", 20, 0, "weekday", -240),
        ("2024-02-29", 19, 0, "weekday", -300),
        ("2024-03-10", 1, 0, "weekend", -300),
        ("2024-03-10", 3, 0, "weekend", -240),
        ("2024-01-15", 5, 30, "weekday", 330),
        ("2024-01-02", 1, 0, "weekday", 780),
    ]
    observed = list(actual[["local_date", "local_hour", "local_minute", "local_day_type", "utc_offset_minutes"]].itertuples(index=False, name=None))
    if observed != expected:
        raise ValueError(f"Node-local clock validation failed: {list(zip(actual['case'], observed, expected))}")

validate_node_clock_examples()

## Validate coverage and physical bounds

Check complete hourly coverage, unique UTC keys and six send and receive candidates per location and UTC day. Verify the local-clock joins, including Kolkata's equal-overlap calculation and matching shape factors for equivalent local times. Check model-layer coverage, complete input fields and the load, headroom and daily-bound constraints.


In [14]:
if len(renewable_model) != len(renewable):
    raise ValueError("Local-clock attachment must preserve the current renewable input row count")

expected_hours = {
    year: int((pd.Timestamp(year=year + 1, month=1, day=1) - pd.Timestamp(year=year, month=1, day=1)) / pd.Timedelta(hours=1))
    for year in ACTIVE_WEATHER_YEARS
}

hours_per_location_year = renewable_model.groupby(["query_location_id", "weather_year"]).size()
for weather_year, expected_hour_count in expected_hours.items():
    expected_groups = pd.MultiIndex.from_product(
        [renewable_model["query_location_id"].unique(), [weather_year]],
        names=["query_location_id", "weather_year"],
    )
    observed_counts = hours_per_location_year.reindex(expected_groups)
    if not observed_counts.eq(expected_hour_count).all():
        raise ValueError(f"Weather year {weather_year} must contain every UTC hour for each query location")

if not renewable_model["utc_date"].eq(renewable_model["time_utc"].dt.date.astype(str)).all():
    raise ValueError("UTC-date boundary changed during local-clock attachment")
if not renewable_model["time_utc"].dt.floor("h").eq(renewable_model["time_utc"]).all():
    raise ValueError("Renewable timestamps must identify exact UTC hours")
if not renewable_model["weather_year"].eq(renewable_model["time_utc"].dt.year).all():
    raise ValueError("Renewable weather-year labels must agree with UTC timestamps")
daily_candidate_counts = renewable_model.groupby(
    ["query_location_id", "weather_year", "utc_date"]
)[["send_candidate", "receive_candidate"]].sum()
if not daily_candidate_counts.eq(6).all().all():
    raise ValueError("Ranked input must identify six send and six receive candidates per UTC day")
if renewable_model.duplicated(["query_location_id", "weather_year", "time_utc"]).any():
    raise ValueError("Renewable model contains duplicate query-location UTC keys")
if not renewable_model.loc[renewable_model["timezone"].eq(HALF_HOUR_OVERLAP_TIMEZONE), "load_profile_join_method"].eq("half_hour_overlap_50_50").all():
    raise ValueError("Asia/Kolkata rows must use the 50/50 overlap rule")
if not renewable_model.loc[~renewable_model["timezone"].eq(HALF_HOUR_OVERLAP_TIMEZONE), "load_profile_join_method"].eq("single_local_hour").all():
    raise ValueError("Integral-offset timezones unexpectedly used an overlap rule")

profile_lookup_validation = profile.set_index(["profile_month", "profile_day_type", "profile_hour"])["normalized_load_shape"].sort_index()
kolkata_sample = renewable_model.loc[renewable_model["timezone"].eq(HALF_HOUR_OVERLAP_TIMEZONE)].head(128).copy()
if not kolkata_sample.empty:
    kolkata_next_source = kolkata_sample[["time_utc", "timezone"]].copy()
    kolkata_next_source["time_utc"] = kolkata_next_source["time_utc"] + pd.Timedelta(minutes=30)
    kolkata_next = add_node_local_clock_fields(kolkata_next_source)
    kolkata_primary_shape = lookup_profile_shape(profile_lookup_validation, kolkata_sample)
    kolkata_secondary_shape = lookup_profile_shape(profile_lookup_validation, kolkata_next)
    if not np.allclose(kolkata_sample["normalized_load_shape"], 0.5 * (kolkata_primary_shape + kolkata_secondary_shape), atol=1e-12, rtol=0):
        raise ValueError("Asia/Kolkata overlap-weight validation failed")

expected_layers = {"current_only", "current_plus_future"}
actual_layers = set(hourly["model_layer"].unique())
if actual_layers != expected_layers:
    raise ValueError(f"Expected model layers {expected_layers}, got {actual_layers}")

hourly_key = ["model_layer", "query_location_id", "operator_group", "weather_year", "time_utc"]
if hourly.duplicated(hourly_key).any():
    raise ValueError("Hourly model contains duplicate canonical UTC keys")
if set(hourly["load_profile_time_basis"].unique()) != {LOAD_PROFILE_TIME_BASIS}:
    raise ValueError("Hourly output does not identify the node-local load-profile time basis")

single_hour_rows = hourly[hourly["load_profile_join_method"].eq("single_local_hour")]
clock_shape_spread = (
    single_hour_rows.groupby(["local_month", "local_day_type", "local_hour"])["normalized_load_shape"]
    .agg(lambda values: float(values.max() - values.min()))
    .max()
)
if clock_shape_spread > 1e-12:
    raise ValueError("Equivalent node-local profile clock keys received different factors")

years = sorted(hourly["weather_year"].dropna().astype(int).unique().tolist())
if years != ACTIVE_WEATHER_YEARS:
    raise ValueError(f"Expected weather years {ACTIVE_WEATHER_YEARS}, got {years}")

required_ranking_cols = ["renewable_availability_index", "receive_rank_daily", "send_rank_daily", "receive_candidate", "send_candidate"]
missing_ranking_rows = hourly[required_ranking_cols].isna().any(axis=1).sum()
if missing_ranking_rows:
    raise ValueError(f"{missing_ranking_rows:,} rows are missing renewable ranking fields")

if hourly["normalized_load_shape"].isna().any():
    raise ValueError("Missing load-shape values after profile join")
if hourly["actual_total_load_mw"].gt(hourly["capacity_mw"] + 1e-9).any():
    raise ValueError("Actual total load exceeds installed capacity")
if hourly["receive_headroom_mw"].lt(-1e-9).any():
    raise ValueError("Receive headroom is negative")
if daily["daily_balanced_shiftable_mwh"].gt(daily["daily_send_candidate_mwh"] + 1e-9).any():
    raise ValueError("Daily balanced shiftable MWh exceeds send candidate MWh")
if daily["daily_balanced_shiftable_mwh"].gt(daily["daily_receive_candidate_mwh"] + 1e-9).any():
    raise ValueError("Daily balanced shiftable MWh exceeds receive candidate MWh")

## Export model inputs and scenarios

Save the hourly input as compressed Parquet and the daily diagnostics and scenario tables as CSV. Retain the defined floating-point representation and display the daily row counts for each model layer.


In [15]:
# Use the defined decimal representation for floating-point export values.
for column in hourly.select_dtypes(include=["float"]).columns:
    hourly[column] = pd.to_numeric(hourly[column].astype(str), errors="raise")

hourly.to_parquet(HOURLY_OUTPUT_PATH, index=False, compression="zstd")
daily.to_csv(DAILY_OUTPUT_PATH, index=False)

FLEXIBILITY_SCENARIOS.to_csv(SCENARIOS_OUTPUT_PATH, index=False)
UTILISATION_SCENARIOS.to_csv(UTILISATION_SCENARIOS_OUTPUT_PATH, index=False)

print("Daily rows by model layer:")
print(daily["model_layer"].value_counts().to_string())

Daily rows by model layer:
model_layer
current_plus_future    119464
current_only            85488
